**[Run this NB in Google Colab](https://colab.research.google.com/github/snad-space/coniferest/blob/master/docs/notebooks/onnx_serialization.ipynb)**

# Saving and loading a session model with ONNX

This notebook shows how to use the `SaveToOnnx` callback to periodically save the model of a running `Session` to ONNX format, and how to load the saved ONNX model back and use it to score new data.

In [ ]:
## Install and import the required libraries

In [4]:
# Install packages

%pip install onnxruntime

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import numpy as np
import onnxruntime as rt
import os
from coniferest.datasets import single_outlier
from coniferest.isoforest import IsolationForest
from coniferest.session import Session
from coniferest.session.callback import Label, SaveToOnnx, TerminateAfter

## Prepare a small dataset

We use the `single_outlier` toy dataset that ships with coniferest.

In [9]:
data, metadata = single_outlier()
data = data.astype(np.float32)

print(f"Data shape: {data.shape}")

Data shape: (10001, 2)


## Run a session that saves the model to ONNX

We attach `SaveToOnnx` as an `on_decision_callback`. Here we save every 5 decisions, keeping a numbered file for each save (`overwrite=False`), so we can inspect how the model evolves over time.

In [10]:
ONNX_DIR = "onnx_models"
EXPERT_BUDGET = 20


def decision(index, x, session):
    # Non-interactive: pick a fixed answer just to drive the demo session
    return Label.REGULAR


model = IsolationForest(
    n_trees=100,
    random_seed=0,
)

session = Session(
    data=data,
    metadata=np.arange(len(data)),
    model=model,
    decision_callback=decision,
    on_decision_callbacks=[
        TerminateAfter(EXPERT_BUDGET),
        SaveToOnnx(
            directory=ONNX_DIR,
            filename="model.onnx",
            every_n_decisions=5,
            overwrite=False,
        ),
    ],
)

session.model.fit(data)
session.run()

The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.


Let's see which ONNX snapshots were saved during the session.

In [12]:
saved_files = sorted(os.listdir(ONNX_DIR))
print(saved_files)

['model_10.onnx', 'model_15.onnx', 'model_20.onnx', 'model_5.onnx']


## Load a saved ONNX model and use it

Pick the last saved snapshot and load it with `onnxruntime` to run inference, without needing the original coniferest model object.

In [13]:
last_snapshot_path = os.path.join(ONNX_DIR, saved_files[-1])

sess = rt.InferenceSession(last_snapshot_path)
input_name = sess.get_inputs()[0].name
label_name = "score"

onnx_scores = sess.run([label_name], {input_name: data})[0].reshape(-1)

print(f"Top 5 most anomalous scores (ONNX): {np.sort(onnx_scores)[:5]}")

Top 5 most anomalous scores (ONNX): [-0.76711404 -0.7537436  -0.75035185 -0.74765104 -0.74630296]


## Overwrite strategy example

If you only care about the latest model and don't want to keep every snapshot, use `overwrite=True` (the default) so the callback always writes to the same file.

In [14]:
OVERWRITE_DIR = "onnx_model_latest"

callback = SaveToOnnx(directory=OVERWRITE_DIR, filename="model.onnx")

session_overwrite = Session(
    data=data,
    metadata=np.arange(len(data)),
    model=IsolationForest(n_trees=100, random_seed=0),
    decision_callback=decision,
    on_decision_callbacks=[
        TerminateAfter(EXPERT_BUDGET),
        callback,
    ],
)
session_overwrite.model.fit(data)
session_overwrite.run()

print(os.listdir(OVERWRITE_DIR))

The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.
The maximum opset needed by this model is only 8.


['model.onnx']
